# 02 - ESM-2 Zero-Shot Mutation Scoring

This notebook loads **ESM-2** (a protein language model trained on evolutionary sequence data) and scores every possible single-point substitution at every position of the chain A reference sequence, using the **masked-marginal** method from Meier et al. (2021), *"Language models enable zero-shot prediction of the effects of mutations on protein function"* (NeurIPS 2021).

**Idea in one line:** mask each position, ask the model "what amino acid belongs here, given everything else in the sequence?", and compare how likely the model thinks the mutant is versus the wild-type residue. This is a proxy for *evolutionary tolerance* - not a direct stability (ΔΔG) estimate. That's what notebook 04 (MACE, a neural network potential) will add as a physics-based, complementary signal.

**Run this notebook in Google Colab with a GPU runtime:** `Runtime` → `Change runtime type` → `T4 GPU`. It has not been executed locally — this project's local `.venv` deliberately stays lightweight (see `requirements.txt`); the heavy ML dependencies below install inline, only inside Colab.

### Running in Colab: mount Google Drive

This notebook needs `data/3eca_chainA.fasta` (from notebook 01) and writes to `results/`. In Colab, the session filesystem is empty by default, so we mount Google Drive and switch into the project folder there - that makes the relative paths (`../data`, `../results`) used below resolve exactly like they do locally.

**Before running:** upload the whole project folder (`data/`, `notebooks/`, `results/`) to your Google Drive, e.g. to `MyDrive/enzyme-design-ai/`. If you used a different folder name/location, edit `PROJECT_DIR` below to match.

In [ ]:
import os

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    PROJECT_DIR = "/content/drive/MyDrive/enzyme-design-ai"  # edit if your folder differs
    os.chdir(f"{PROJECT_DIR}/notebooks")
    print(f"Working directory: {os.getcwd()}")
else:
    print("Not running in Colab — skipping Drive mount, using local relative paths.")

## Setup

In [ ]:
%pip install -q fair-esm

In [ ]:
import torch
import esm
import pandas as pd
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cpu":
    print("WARNING: no GPU detected. Go to Runtime > Change runtime type > T4 GPU, "
          "then re-run — this will be very slow on CPU.")

## Load the reference sequence

Reads the FASTA saved by notebook 01. No Biopython dependency needed here - it's a plain two-part FASTA file, so a couple of lines are enough.

In [ ]:
def read_fasta(path: Path) -> tuple[str, str]:
    lines = Path(path).read_text().splitlines()
    header = lines[0].lstrip(">")
    sequence = "".join(lines[1:])
    return header, sequence


DATA_DIR = Path("../data")
header, sequence = read_fasta(DATA_DIR / "3eca_chainA.fasta")

print(header)
print(f"Reference sequence: {len(sequence)} residues")

# Sanity check against notebook 01's confirmed catalytic residues (1-indexed, matches PDB numbering)
assert sequence[12 - 1] == "T", "Expected Thr at position 12"
assert sequence[89 - 1] == "T", "Expected Thr at position 89"
print("Catalytic residue positions (Thr12, Thr89) confirmed in the loaded sequence.")

## Load ESM-2

Using `esm2_t33_650M_UR50D` - a good quality/speed balance on a free T4 GPU.

In [ ]:
model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
model.eval()
model = model.to(device)

batch_converter = alphabet.get_batch_converter()
n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded ESM-2 ({n_params:,} parameters) on {device}")

## Masked-marginal scoring

For each position *i*: replace the residue with a `<mask>` token, run a forward pass, and read off the model's predicted log-probability for every amino acid at that position. The score for a substitution `wt -> mut` at position *i* is:

```
score(wt -> mut) = log P(mut | sequence, position i masked) - log P(wt | sequence, position i masked)
```

Positive = the model finds the mutant *more* natural in that context than the wild-type residue (weak evidence of tolerance, not proof). Negative = the model prefers the wild-type — mutation likely deleterious.

Masking (rather than just reading probabilities off a single unmasked pass) matters: it stops the model from "cheating" by simply recognizing the wild-type token already sitting in the sequence, which the paper found gives noticeably better agreement with experimental mutation effects.

We batch several masked positions together per forward pass — one-by-one would mean 326 separate forward passes, which is needlessly slow on a GPU.

In [ ]:
BATCH_SIZE = 8  # tune down if you hit a CUDA out-of-memory error

mask_idx = alphabet.mask_idx
_, _, base_tokens = batch_converter([("wt", sequence)])
base_tokens = base_tokens.to(device)  # shape [1, L + 2] (BOS + L residues + EOS)

all_log_probs = torch.zeros(len(sequence), len(alphabet))

with torch.no_grad():
    for start in range(0, len(sequence), BATCH_SIZE):
        positions = list(range(start, min(start + BATCH_SIZE, len(sequence))))
        batch_tokens = base_tokens.repeat(len(positions), 1).clone()
        for row, pos in enumerate(positions):
            batch_tokens[row, pos + 1] = mask_idx  # +1: skip the BOS token at index 0

        logits = model(batch_tokens)["logits"]
        for row, pos in enumerate(positions):
            all_log_probs[pos] = torch.log_softmax(logits[row, pos + 1], dim=-1).cpu()

        if start % (BATCH_SIZE * 10) == 0:
            print(f"Scored {start + len(positions)}/{len(sequence)} positions")

print("Done.")
print(all_log_probs.shape)

## Build the ranked mutation table

In [ ]:
AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"

rows = []
for i, wt_aa in enumerate(sequence):
    pos = i + 1  # 1-indexed, matches PDB residue numbering (confirmed above)
    wt_log_prob = all_log_probs[i, alphabet.get_idx(wt_aa)].item()
    for mut_aa in AMINO_ACIDS:
        if mut_aa == wt_aa:
            continue
        mut_log_prob = all_log_probs[i, alphabet.get_idx(mut_aa)].item()
        rows.append({
            "position": pos,
            "wt": wt_aa,
            "mut": mut_aa,
            "mutation": f"{wt_aa}{pos}{mut_aa}",
            "esm_score": mut_log_prob - wt_log_prob,
        })

scores_df = pd.DataFrame(rows).sort_values("esm_score", ascending=False).reset_index(drop=True)
print(f"{len(scores_df)} candidate substitutions scored ({len(sequence)} positions x 19 alternates)")
scores_df.head(20)

## Flag mutations at the known catalytic residues

As established in notebook 01: the active site is formed *between* subunits, with Thr12 and Thr89 as known catalytic residues. A high ESM-2 score there is not trustworthy - the model has no notion of "this position matters for chemistry, not just fold stability." We flag the two known catalytic positions directly here.

The *full* inter-subunit interface (which residues actually sit at the A/B, A/C, A/D contacts) needs 3D contact analysis, which requires a folded/complexed structure - that's deferred to notebook 03, once we have coordinates to measure distances on.

In [ ]:
CATALYTIC_POSITIONS = {12, 89}
scores_df["at_catalytic_residue"] = scores_df["position"].isin(CATALYTIC_POSITIONS)

n_flagged = int(scores_df["at_catalytic_residue"].sum())
print(f"{n_flagged} candidate substitutions sit directly on a catalytic residue (Thr12/Thr89) "
      "and should be excluded or manually scrutinized regardless of ESM-2 score.")

## Save results for notebook 03

In [ ]:
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)

scores_path = RESULTS_DIR / "esm2_mutation_scores.csv"
scores_df.to_csv(scores_path, index=False)
print(f"Saved full scoring table ({len(scores_df)} rows) to {scores_path}")

top_candidates = scores_df[~scores_df["at_catalytic_residue"]].head(20)
print("\nTop 20 candidates (catalytic-residue positions excluded):")
top_candidates

## Summary

- Loaded ESM-2 (650M params) and computed masked-marginal scores for all possible single-point substitutions across the 326-residue chain A sequence.
- Higher `esm_score` = the model considers the substitution more evolutionarily natural at that position - a proxy signal, not a direct stability or activity measurement.
- Flagged substitutions landing on the known catalytic residues (Thr12, Thr89) for exclusion/extra scrutiny.
- Saved the full table to `results/esm2_mutation_scores.csv`.
- **Next:** `03_structure_prediction.ipynb` - fold the wild type and top candidate variants with ESMFold, check fold quality (pLDDT), and run a proper 3D contact analysis to flag any candidate sitting near the inter-subunit interface (beyond just the two known catalytic positions).